# Day 19 — Shortlisted Candidate Tuning

## Objective

The objective of Day 19 is to systematically tune the shortlisted anomaly-detection and forecasting candidates using documented search strategies.

## Shortlisted Candidates

### Anomaly Detection
- Isolation Forest

### Forecasting
- Random Forest Regressor

## Tuning Strategy

Meaningful hyperparameters will be changed systematically across predefined candidate values.

### Isolation Forest
- Number of estimators
- Contamination
- Maximum samples

### Random Forest
- Number of estimators
- Maximum depth
- Minimum samples required for a split
- Minimum samples required at a leaf

## Validation Policy

- Training data is used for model fitting.
- Validation data is used for hyperparameter comparison and selection.
- The final test set remains untouched during tuning.
- The selected configuration will be retrained only after the validation-based selection.

## Experiment Logging

Every experiment will record:
- Experiment ID
- Model
- Hyperparameter configuration
- Validation metric
- Training time
- Selection status

## Output

The final outputs of Day 19 will be:

1. Complete experiment log
2. Best validated Isolation Forest configuration
3. Best validated Random Forest configuration
4. Saved model configurations
5. Documentation of the search strategy

In [1]:
import pandas as pd
import numpy as np
import joblib
import time

from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

base_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed"

# Load datasets
train_df = pd.read_csv(base_path + r"\train_v1_engineered.csv")
val_df = pd.read_csv(base_path + r"\validation_v1_engineered.csv")
test_df = pd.read_csv(base_path + r"\test_v1_engineered.csv")

# Convert timestamps
for data in [train_df, val_df, test_df]:
    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        errors="coerce"
    )
    data.sort_values("timestamp", inplace=True)
    data.reset_index(drop=True, inplace=True)

sensor_cols = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity"
]

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Sensor features:", len(sensor_cols))

Train shape: (2291, 42)
Validation shape: (491, 42)
Test shape: (491, 42)
Sensor features: 9


In [2]:
# Prepare anomaly detection datasets

X_train_anomaly = train_df[sensor_cols]
X_val_anomaly = val_df[sensor_cols]

# Fit imputer ONLY on training data
anomaly_imputer = SimpleImputer(strategy="median")

X_train_imp = anomaly_imputer.fit_transform(X_train_anomaly)
X_val_imp = anomaly_imputer.transform(X_val_anomaly)

print("Training anomaly data:", X_train_imp.shape)
print("Validation anomaly data:", X_val_imp.shape)

Training anomaly data: (2291, 9)
Validation anomaly data: (491, 9)


## Isolation Forest — Search Strategy

A small systematic grid search will be used rather than changing parameters randomly.

The following configurations will be evaluated:

- `n_estimators`: 100, 200, 300
- `contamination`: 0.03, 0.05, 0.10
- `max_samples`: "sqrt", 0.8, 1.0

This produces a controlled set of candidate configurations.

The validation anomaly rate will be recorded for every experiment.

The test set will not be used during this search.

In [4]:
isolation_configs = []

for n_estimators in [100, 200, 300]:
    for contamination in [0.03, 0.05, 0.10]:
        for max_samples in ["sqrt", 0.8, 1.0]:
            isolation_configs.append({
                "n_estimators": n_estimators,
                "contamination": contamination,
                "max_samples": max_samples
            })

print("Total Isolation Forest experiments:", len(isolation_configs))

Total Isolation Forest experiments: 27


In [6]:
isolation_results = []

for i, config in enumerate(isolation_configs, start=1):

    start_time = time.perf_counter()

    # Use an integer for max_samples because it is supported
    # consistently across scikit-learn versions.
    max_samples_value = config["max_samples"]

    if max_samples_value == "sqrt":
        max_samples_value = min(512, X_train_imp.shape[0])

    model = IsolationForest(
        n_estimators=config["n_estimators"],
        contamination=config["contamination"],
        max_samples=max_samples_value,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_imp)

    train_predictions = model.predict(X_train_imp)
    val_predictions = model.predict(X_val_imp)

    train_anomaly_rate = np.mean(train_predictions == -1) * 100
    val_anomaly_rate = np.mean(val_predictions == -1) * 100

    elapsed_time = time.perf_counter() - start_time

    isolation_results.append({
        "Experiment ID": f"IF_{i:02d}",
        "Model": "Isolation Forest",
        "n_estimators": config["n_estimators"],
        "contamination": config["contamination"],
        "max_samples": max_samples_value,
        "Train Anomaly Rate (%)": round(train_anomaly_rate, 2),
        "Validation Anomaly Rate (%)": round(val_anomaly_rate, 2),
        "Rate Difference (%)": round(
            abs(train_anomaly_rate - val_anomaly_rate), 2
        ),
        "Processing Time (sec)": round(elapsed_time, 4)
    })

isolation_results_df = pd.DataFrame(isolation_results)

print("Experiments completed:", len(isolation_results_df))

isolation_results_df

Experiments completed: 27


,Experiment ID,Model,n_estimators,contamination,max_samples,Train Anomaly Rate (%),Validation Anomaly Rate (%),Rate Difference (%),Processing Time (sec)
0,IF_01,Isolation Forest,100,0.03,512.0,3.01,3.05,0.04,0.4603
1,IF_02,Isolation Forest,100,0.03,0.8,3.01,3.26,0.25,0.4204
2,IF_03,Isolation Forest,100,0.03,1.0,3.01,3.05,0.04,0.4378
3,IF_04,Isolation Forest,100,0.05,512.0,5.02,5.91,0.89,0.3846
4,IF_05,Isolation Forest,100,0.05,0.8,5.02,6.31,1.29,0.3823
5,IF_06,Isolation Forest,100,0.05,1.0,5.02,5.09,0.07,0.3677
6,IF_07,Isolation Forest,100,0.10,512.0,10.00,14.26,4.26,0.2912
7,IF_08,Isolation Forest,100,0.10,0.8,10.00,13.85,3.85,0.2832
8,IF_09,Isolation Forest,100,0.10,1.0,10.00,13.65,3.65,0.3054
9,IF_10,Isolation Forest,200,0.03,512.0,3.01,3.05,0.04,0.7287


## Isolation Forest — Validation-Based Selection

The best configuration is selected using the validation anomaly rate.

Since ground-truth anomaly labels are unavailable, the primary selection criterion is the stability of anomaly rates between training and validation data.

A smaller train-validation anomaly-rate difference indicates more stable behaviour.

The test set remains untouched.

In [7]:
# Select the most stable Isolation Forest configuration

best_if_row = isolation_results_df.sort_values(
    "Rate Difference (%)"
).iloc[0]

best_if_row

Experiment ID                             IF_01
Model                          Isolation Forest
n_estimators                                100
contamination                              0.03
max_samples                               512.0
Train Anomaly Rate (%)                     3.01
Validation Anomaly Rate (%)                3.05
Rate Difference (%)                        0.04
Processing Time (sec)                    0.4603
Name: 0, dtype: object

In [8]:
# Extract best Isolation Forest configuration

best_if_config = {
    "n_estimators": int(best_if_row["n_estimators"]),
    "contamination": float(best_if_row["contamination"]),
    "max_samples": int(best_if_row["max_samples"])
}

print("Best Isolation Forest configuration:")
print(best_if_config)

Best Isolation Forest configuration:
{'n_estimators': 100, 'contamination': 0.03, 'max_samples': 512}


## Random Forest — Search Strategy

The Random Forest forecasting candidate will be tuned using a controlled grid search.

The following meaningful hyperparameters will be varied:

- `n_estimators`: 100, 200, 300
- `max_depth`: 5, 10, None
- `min_samples_split`: 2, 5
- `min_samples_leaf`: 1, 2

Validation MAE and RMSE will be used to compare configurations.

The test set will remain completely untouched during tuning.

In [9]:
forecast_features = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity",
    "hour",
    "day_of_week",
    "day",
    "month",
    "ph_lag_1",
    "ph_rolling_mean_3",
    "ph_rolling_std_3"
]

target_col = "ph"

# Create next-step target
train_forecast = train_df.copy()
val_forecast = val_df.copy()

train_forecast["ph_next"] = train_forecast[target_col].shift(-1)
val_forecast["ph_next"] = val_forecast[target_col].shift(-1)

train_forecast = train_forecast.dropna(
    subset=["ph_next"]
).reset_index(drop=True)

val_forecast = val_forecast.dropna(
    subset=["ph_next"]
).reset_index(drop=True)

X_train_rf = train_forecast[forecast_features]
y_train_rf = train_forecast["ph_next"]

X_val_rf = val_forecast[forecast_features]
y_val_rf = val_forecast["ph_next"]

# Fit imputer ONLY on training data
rf_imputer = SimpleImputer(strategy="median")

X_train_rf_imp = rf_imputer.fit_transform(X_train_rf)
X_val_rf_imp = rf_imputer.transform(X_val_rf)

print("Training forecasting data:", X_train_rf_imp.shape)
print("Validation forecasting data:", X_val_rf_imp.shape)

Training forecasting data: (2290, 16)
Validation forecasting data: (490, 16)


In [10]:
rf_configs = []

for n_estimators in [100, 200, 300]:
    for max_depth in [5, 10, None]:
        for min_samples_split in [2, 5]:
            for min_samples_leaf in [1, 2]:

                rf_configs.append({
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "min_samples_split": min_samples_split,
                    "min_samples_leaf": min_samples_leaf
                })

print("Total Random Forest experiments:", len(rf_configs))

Total Random Forest experiments: 36


In [11]:
rf_results = []

for i, config in enumerate(rf_configs, start=1):

    start_time = time.perf_counter()

    model = RandomForestRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_split=config["min_samples_split"],
        min_samples_leaf=config["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_rf_imp, y_train_rf)

    val_predictions = model.predict(X_val_rf_imp)

    val_mae = mean_absolute_error(
        y_val_rf,
        val_predictions
    )

    val_rmse = np.sqrt(
        mean_squared_error(
            y_val_rf,
            val_predictions
        )
    )

    elapsed_time = time.perf_counter() - start_time

    rf_results.append({
        "Experiment ID": f"RF_{i:02d}",
        "Model": "Random Forest",
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "min_samples_split": config["min_samples_split"],
        "min_samples_leaf": config["min_samples_leaf"],
        "Validation MAE": round(val_mae, 6),
        "Validation RMSE": round(val_rmse, 6),
        "Processing Time (sec)": round(elapsed_time, 4)
    })

rf_results_df = pd.DataFrame(rf_results)

print("Experiments completed:", len(rf_results_df))

rf_results_df

Experiments completed: 36


,Experiment ID,Model,n_estimators,max_depth,min_samples_split,min_samples_leaf,Validation MAE,Validation RMSE,Processing Time (sec)
0,RF_01,Random Forest,100,5.0,2,1,0.831742,1.042586,0.4257
1,RF_02,Random Forest,100,5.0,2,2,0.830124,1.041976,0.3327
2,RF_03,Random Forest,100,5.0,5,1,0.830581,1.041370,0.3753
3,RF_04,Random Forest,100,5.0,5,2,0.829904,1.041710,0.3926
4,RF_05,Random Forest,100,10.0,2,1,0.836660,1.048355,0.5513
5,RF_06,Random Forest,100,10.0,2,2,0.835454,1.049337,0.5291
6,RF_07,Random Forest,100,10.0,5,1,0.838213,1.049069,0.4564
7,RF_08,Random Forest,100,10.0,5,2,0.837344,1.050440,0.5502
8,RF_09,Random Forest,100,NaN,2,1,0.845569,1.058102,0.7021
9,RF_10,Random Forest,100,NaN,2,2,0.843228,1.055470,0.7729


## Random Forest — Validation-Based Selection

The Random Forest configuration with the lowest validation MAE is selected.

Validation RMSE is retained as a secondary performance metric.

The test set remains untouched and will only be used in a later final evaluation stage.

In [12]:
# Select best Random Forest using validation MAE

best_rf_row = rf_results_df.sort_values(
    "Validation MAE"
).iloc[0]

best_rf_row

Experiment ID                    RF_28
Model                    Random Forest
n_estimators                       300
max_depth                          5.0
min_samples_split                    5
min_samples_leaf                     2
Validation MAE                0.828068
Validation RMSE               1.039901
Processing Time (sec)           0.8781
Name: 27, dtype: object

In [13]:
best_rf_config = {
    "n_estimators": int(best_rf_row["n_estimators"]),
    "max_depth": (
        None
        if pd.isna(best_rf_row["max_depth"])
        else int(best_rf_row["max_depth"])
    ),
    "min_samples_split": int(best_rf_row["min_samples_split"]),
    "min_samples_leaf": int(best_rf_row["min_samples_leaf"])
}

print("Best Random Forest configuration:")
print(best_rf_config)

print("\nValidation MAE:",
      best_rf_row["Validation MAE"])

print("Validation RMSE:",
      best_rf_row["Validation RMSE"])

Best Random Forest configuration:
{'n_estimators': 300, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 2}

Validation MAE: 0.828068
Validation RMSE: 1.039901


In [14]:
# Combine all Day 19 experiments into one experiment log

experiment_log = pd.concat(
    [
        isolation_results_df,
        rf_results_df
    ],
    ignore_index=True,
    sort=False
)

print("Total experiments:", len(experiment_log))
experiment_log

Total experiments: 63


,Experiment ID,Model,n_estimators,contamination,max_samples,Train Anomaly Rate (%),Validation Anomaly Rate (%),Rate Difference (%),Processing Time (sec),max_depth,min_samples_split,min_samples_leaf,Validation MAE,Validation RMSE
0,IF_01,Isolation Forest,100,0.03,512.0,3.01,3.05,0.04,0.4603,NaN,NaN,NaN,NaN,NaN
1,IF_02,Isolation Forest,100,0.03,0.8,3.01,3.26,0.25,0.4204,NaN,NaN,NaN,NaN,NaN
2,IF_03,Isolation Forest,100,0.03,1.0,3.01,3.05,0.04,0.4378,NaN,NaN,NaN,NaN,NaN
3,IF_04,Isolation Forest,100,0.05,512.0,5.02,5.91,0.89,0.3846,NaN,NaN,NaN,NaN,NaN
4,IF_05,Isolation Forest,100,0.05,0.8,5.02,6.31,1.29,0.3823,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,RF_32,Random Forest,300,NaN,NaN,NaN,NaN,NaN,1.4486,10.0,5.0,2.0,0.833141,1.045107
59,RF_33,Random Forest,300,NaN,NaN,NaN,NaN,NaN,2.0738,NaN,2.0,1.0,0.841955,1.054699
60,RF_34,Random Forest,300,NaN,NaN,NaN,NaN,NaN,1.4860,NaN,2.0,2.0,0.837343,1.048090
61,RF_35,Random Forest,300,NaN,NaN,NaN,NaN,NaN,1.5866,NaN,5.0,1.0,0.840957,1.053514


In [15]:
import os

project_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics"

experiments_path = os.path.join(
    project_path,
    "experiments"
)

models_path = os.path.join(
    project_path,
    "models"
)

os.makedirs(experiments_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)

print("Experiments folder:", experiments_path)
print("Models folder:", models_path)

Experiments folder: C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\experiments
Models folder: C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\models


In [16]:
experiment_log_path = os.path.join(
    experiments_path,
    "day19_experiment_log.csv"
)

experiment_log.to_csv(
    experiment_log_path,
    index=False
)

print("Experiment log saved:")
print(experiment_log_path)

Experiment log saved:
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\experiments\day19_experiment_log.csv


In [17]:
# Retrain best Isolation Forest configuration on training data

best_if_model = IsolationForest(
    n_estimators=best_if_config["n_estimators"],
    contamination=best_if_config["contamination"],
    max_samples=best_if_config["max_samples"],
    random_state=42,
    n_jobs=-1
)

best_if_model.fit(X_train_imp)

best_if_model_path = os.path.join(
    models_path,
    "day19_best_isolation_forest.joblib"
)

joblib.dump(
    {
        "model": best_if_model,
        "imputer": anomaly_imputer,
        "features": sensor_cols,
        "configuration": best_if_config,
        "selection_metric": "Validation anomaly-rate stability"
    },
    best_if_model_path
)

print("Best Isolation Forest saved:")
print(best_if_model_path)

Best Isolation Forest saved:
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\models\day19_best_isolation_forest.joblib


In [18]:
# Retrain best Random Forest configuration on training data

best_rf_model = RandomForestRegressor(
    n_estimators=best_rf_config["n_estimators"],
    max_depth=best_rf_config["max_depth"],
    min_samples_split=best_rf_config["min_samples_split"],
    min_samples_leaf=best_rf_config["min_samples_leaf"],
    random_state=42,
    n_jobs=-1
)

best_rf_model.fit(
    X_train_rf_imp,
    y_train_rf
)

best_rf_model_path = os.path.join(
    models_path,
    "day19_best_random_forest.joblib"
)

joblib.dump(
    {
        "model": best_rf_model,
        "imputer": rf_imputer,
        "features": forecast_features,
        "target": "ph_next",
        "configuration": best_rf_config,
        "selection_metric": "Validation MAE"
    },
    best_rf_model_path
)

print("Best Random Forest saved:")
print(best_rf_model_path)

Best Random Forest saved:
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\models\day19_best_random_forest.joblib


# Day 19 — Final Summary

## Experiment Summary

A systematic hyperparameter search was performed for both shortlisted candidates.

### Isolation Forest
- 27 configurations evaluated.
- Parameters varied:
  - `n_estimators`
  - `contamination`
  - `max_samples`
- Training and validation anomaly rates were recorded.
- Validation anomaly-rate stability was used for selection.

### Random Forest
- 36 configurations evaluated.
- Parameters varied:
  - `n_estimators`
  - `max_depth`
  - `min_samples_split`
  - `min_samples_leaf`
- Validation MAE and RMSE were recorded.
- Lowest validation MAE was used for selection.

## Validation Policy

The test dataset was not used during hyperparameter tuning or model selection.

Training data was used for fitting, while validation data was used exclusively for comparing configurations.

## Saved Outputs

- `experiments/day19_experiment_log.csv`
- `models/day19_best_isolation_forest.joblib`
- `models/day19_best_random_forest.joblib`

## Limitations

- Ground-truth anomaly labels are unavailable.
- The timestamps are synthetically generated.
- The dataset is relatively small.
- The available dataset does not contain temperature, dissolved oxygen, or pressure.
- Forecasting experiments currently focus on next-step pH prediction.

## Next Step

The selected configurations can be evaluated on the untouched test set during the final evaluation stage.